## **Importing Packages**


In [ ]:
!pip install -q scikit-survival openpyxl


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import set_config
from sklearn.exceptions import FitFailedWarning
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from sksurv.linear_model import CoxnetSurvivalAnalysis

set_config(display="text")
%matplotlib inline


In [ ]:
df = pd.read_csv("Data/final_dataset.csv")
feature_cols = [c for c in df.columns if c not in ("Sample Identifier", "ICB Response")]
expr_feature_cols = [c for c in feature_cols if not c.endswith("_SNP") and not c.endswith("_CN")]
snp_feature_cols = [c for c in feature_cols if c.endswith("_SNP")]


## **Helper Functions**


In [ ]:
'''
A function that merges columns based on a common column.
It takes as an input a list of all the dataframes that should be merged and the merging column
This function is very important as we will use it often later
'''


def merge_on_common_column(dataframes, merge_column):
    merged_df = dataframes[0]
    for df_in in dataframes[1:]:
        merged_df = pd.merge(merged_df, df_in, on=merge_column)
    return merged_df


In [ ]:
def load_liu_survival():
    patient = pd.read_csv("Data/Liu/data_clinical_patient.tsv", sep="\t").iloc[4:]
    sample = pd.read_csv("Data/Liu/data_clinical_sample.tsv", sep="\t").iloc[4:]
    clin = merge_on_common_column([sample, patient], "#Patient Identifier")
    out = clin[
        [
            "Sample Identifier",
            "Overall Survival Status",
            "Overall Survival (Months)",
        ]
    ].copy()
    out["Overall Survival Status"] = out["Overall Survival Status"].map(
        {"1:DECEASED": True, "0:LIVING": False}
    )
    out["Overall Survival (Months)"] = pd.to_numeric(
        out["Overall Survival (Months)"], errors="coerce"
    ).astype(float) * 30.0
    return out


def load_ravi_survival():
    ravi_xlsx = "Data/Ravi/clinical.xlsx"
    if not ravi_xlsx.exists():
        return pd.DataFrame(
            columns=[
                "Sample Identifier",
                "Overall Survival Status",
                "Overall Survival (Months)",
            ]
        )
    clinical1_ravi = pd.read_excel(ravi_xlsx)
    clinical1_ravi["Harmonized_SU2C_WES_Tumor_Sample_ID_v2"] = (
        clinical1_ravi["Harmonized_SU2C_WES_Tumor_Sample_ID_v2"].str.replace("-T", "", regex=False)
    )
    out = clinical1_ravi[
        [
            "Harmonized_SU2C_WES_Tumor_Sample_ID_v2",
            "Harmonized_OS_Event",
            "Harmonized_OS_Days",
        ]
    ].rename(
        columns={
            "Harmonized_SU2C_WES_Tumor_Sample_ID_v2": "Sample Identifier",
            "Harmonized_OS_Event": "Overall Survival Status",
            "Harmonized_OS_Days": "Overall Survival (Months)",
        }
    )
    out["Overall Survival Status"] = out["Overall Survival Status"].map({1: True, 0: False})
    out["Overall Survival (Months)"] = pd.to_numeric(
        out["Overall Survival (Months)"], errors="coerce"
    )
    return out


df_surv = pd.concat([load_ravi_survival(), load_liu_survival()], ignore_index=True)
df_surv = df_surv.dropna(subset=["Sample Identifier"])


## **Transcriptomic survival matrix**


In [ ]:
merged_expression = pd.read_csv("Data/Merged/merged_expression.tsv", sep="\t")
expr_cols_avail = [c for c in expr_feature_cols if c in merged_expression.columns]
expr_x = merged_expression[["Sample Identifier"] + expr_cols_avail]

merged_surv = merge_on_common_column([df_surv, expr_x], "Sample Identifier").dropna(axis=0)

X_expr = merged_surv.drop(
    columns=["Sample Identifier", "Overall Survival Status", "Overall Survival (Months)"]
)
dtype_surv = np.dtype([("e.tdm", "?"), ("t.tdm", "<f8")])
y_surv = np.array(
    list(
        merged_surv[["Overall Survival Status", "Overall Survival (Months)"]].to_records(
            index=False
        )
    ),
    dtype=dtype_surv,
)
print(len(merged_surv), "samples,", X_expr.shape[1], "expression features")


## **Cox elastic net (transcriptomic)**


In [ ]:
cox_elastic_net = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01)
cox_elastic_net.fit(X_expr, y_surv)
print("concordance index (train):", cox_elastic_net.score(X_expr, y_surv))


In [ ]:
def plot_coefficients(coefs, n_highlight):
    _, ax = plt.subplots(figsize=(9, 6))
    alphas = coefs.columns
    for row in coefs.itertuples():
        ax.semilogx(alphas, row[1:], ".-", label=row.Index)

    alpha_min = alphas.min()
    top_coefs = coefs.loc[:, alpha_min].map(abs).sort_values().tail(n_highlight)
    for name in top_coefs.index:
        coef = coefs.loc[name, alpha_min]
        ax.text(
            alpha_min,
            coef,
            name + "   ",
            horizontalalignment="right",
            verticalalignment="center",
        )

    ax.yaxis.set_label_position("right")
    ax.yaxis.tick_right()
    ax.grid(True)
    ax.set_xlabel("alpha")
    ax.set_ylabel("coefficient")


coefficients_elastic_net = pd.DataFrame(
    cox_elastic_net.coef_,
    index=X_expr.columns,
    columns=np.round(cox_elastic_net.alphas_, 5),
)
plot_coefficients(coefficients_elastic_net, n_highlight=5)
plt.show()


In [ ]:
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FitFailedWarning)

coxnet_pipe = make_pipeline(
    StandardScaler(),
    CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01, max_iter=100),
)
coxnet_pipe.fit(X_expr, y_surv)

estimated_alphas = coxnet_pipe.named_steps["coxnetsurvivalanalysis"].alphas_
cv = KFold(n_splits=5, shuffle=True, random_state=0)
gcv_expr = GridSearchCV(
    make_pipeline(StandardScaler(), CoxnetSurvivalAnalysis(l1_ratio=0.9)),
    param_grid={"coxnetsurvivalanalysis__alphas": [[v] for v in estimated_alphas]},
    cv=cv,
    error_score=0.5,
    n_jobs=1,
).fit(X_expr, y_surv)

cv_results_expr = pd.DataFrame(gcv_expr.cv_results_)

alphas = cv_results_expr.param_coxnetsurvivalanalysis__alphas.map(lambda x: x[0])
mean = cv_results_expr.mean_test_score
std = cv_results_expr.std_test_score

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(alphas, mean)
ax.fill_between(alphas, mean - std, mean + std, alpha=0.15)
ax.set_xscale("log")
ax.set_ylabel("concordance index")
ax.set_xlabel("alpha")
ax.axvline(gcv_expr.best_params_["coxnetsurvivalanalysis__alphas"][0], c="C1")
ax.axhline(0.5, color="grey", linestyle="--")
ax.grid(True)
fig.savefig("parameters_plot.png", dpi=600, bbox_inches="tight")
plt.show()
print("best alpha:", gcv_expr.best_params_["coxnetsurvivalanalysis__alphas"][0])


In [ ]:
best_model_expr = gcv_expr.best_estimator_.named_steps["coxnetsurvivalanalysis"]
best_coefs_expr = pd.DataFrame(
    best_model_expr.coef_, index=X_expr.columns, columns=["coefficient"]
)

non_zero = np.sum(best_coefs_expr.iloc[:, 0] != 0)
print(f"Number of non-zero coefficients: {non_zero}")

non_zero_coefs_expr = best_coefs_expr.query("coefficient != 0")
coef_order = non_zero_coefs_expr.abs().sort_values("coefficient").index

fig, ax = plt.subplots(figsize=(9, 13))
non_zero_coefs_expr.loc[coef_order].plot.barh(ax=ax, legend=False)
ax.set_xlabel("coefficient")
ax.grid(True)
fig.savefig("coefficients_plot.png", dpi=600, bbox_inches="tight")
plt.show()

non_zero_coefs_expr.loc[coef_order].to_csv("transcriptomic_data_survival.csv")


## **SNP survival matrix**


In [ ]:
merged_snp = pd.read_csv("Data/Merged/merged_snp_pre_normalization.tsv", sep="\t")
snp_cols_avail = [c for c in snp_feature_cols if c in merged_snp.columns]
snp_x = merged_snp[["Sample Identifier"] + snp_cols_avail]

merged_surv_snp = merge_on_common_column([df_surv, snp_x], "Sample Identifier").dropna(axis=0)

X_snp = merged_surv_snp.drop(
    columns=["Sample Identifier", "Overall Survival Status", "Overall Survival (Months)"]
)
y_snp = np.array(
    list(
        merged_surv_snp[
            ["Overall Survival Status", "Overall Survival (Months)"]
        ].to_records(index=False)
    ),
    dtype=dtype_surv,
)
print(len(merged_surv_snp), "samples,", X_snp.shape[1], "SNP features")


## **Cox elastic net (SNP)**


In [ ]:
cox_elastic_net_snp = CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01)
cox_elastic_net_snp.fit(X_snp, y_snp)
print("concordance index (train):", cox_elastic_net_snp.score(X_snp, y_snp))


In [ ]:
coefficients_elastic_net_snp = pd.DataFrame(
    cox_elastic_net_snp.coef_,
    index=X_snp.columns,
    columns=np.round(cox_elastic_net_snp.alphas_, 5),
)
plot_coefficients(coefficients_elastic_net_snp, n_highlight=5)
plt.show()


In [ ]:
coxnet_pipe_snp = make_pipeline(
    StandardScaler(),
    CoxnetSurvivalAnalysis(l1_ratio=0.9, alpha_min_ratio=0.01, max_iter=100),
)
coxnet_pipe_snp.fit(X_snp, y_snp)

estimated_alphas_snp = coxnet_pipe_snp.named_steps["coxnetsurvivalanalysis"].alphas_
gcv_snp = GridSearchCV(
    make_pipeline(StandardScaler(), CoxnetSurvivalAnalysis(l1_ratio=0.9)),
    param_grid={"coxnetsurvivalanalysis__alphas": [[v] for v in estimated_alphas_snp]},
    cv=cv,
    error_score=0.5,
    n_jobs=1,
).fit(X_snp, y_snp)

cv_results_snp = pd.DataFrame(gcv_snp.cv_results_)

alphas = cv_results_snp.param_coxnetsurvivalanalysis__alphas.map(lambda x: x[0])
mean = cv_results_snp.mean_test_score
std = cv_results_snp.std_test_score

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(alphas, mean)
ax.fill_between(alphas, mean - std, mean + std, alpha=0.15)
ax.set_xscale("log")
ax.set_ylabel("concordance index")
ax.set_xlabel("alpha")
ax.axvline(gcv_snp.best_params_["coxnetsurvivalanalysis__alphas"][0], c="C1")
ax.axhline(0.5, color="grey", linestyle="--")
ax.grid(True)
fig.savefig("parameters_snp_plot.png", dpi=600, bbox_inches="tight")
plt.show()
print("best alpha:", gcv_snp.best_params_["coxnetsurvivalanalysis__alphas"][0])


In [ ]:
best_model_snp = gcv_snp.best_estimator_.named_steps["coxnetsurvivalanalysis"]
best_coefs_snp = pd.DataFrame(
    best_model_snp.coef_, index=X_snp.columns, columns=["coefficient"]
)

non_zero = np.sum(best_coefs_snp.iloc[:, 0] != 0)
print(f"Number of non-zero coefficients: {non_zero}")

non_zero_coefs_snp = best_coefs_snp.query("coefficient != 0")
coef_order_snp = non_zero_coefs_snp.abs().sort_values("coefficient").index

fig, ax = plt.subplots(figsize=(9, 13))
non_zero_coefs_snp.loc[coef_order_snp].plot.barh(ax=ax, legend=False)
ax.set_xlabel("coefficient")
ax.grid(True)
fig.savefig("coefficients_snp_plot.png", dpi=600, bbox_inches="tight")
plt.show()

non_zero_coefs_snp.loc[coef_order_snp].to_csv("SNP_survival.csv")
